# 🛡️ SafeGuard Vision AI - Video Color Extractor

**Project:** MIT Global Teaching Labs  
**Author:** Grupo Outliers

This notebook extracts the **COLOR** portion of videos that have a split format (binary + color).

---

## 📌 Step 1: Mount Google Drive

Run this cell and authorize access to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")

## 📌 Step 2: Configure Paths

⚠️ **IMPORTANT:** Modify the paths according to your folder structure in Drive.

**Example:** If your folder is in `My Drive > Project > Videos`, the path would be:
```
/content/drive/MyDrive/Project/Videos
```

In [ ]:
# ============================================================
# 🔧 CONFIGURE THESE PATHS (modify according to your Drive)
# ============================================================

# Folder where your original videos are stored in Drive
INPUT_FOLDER = "/content/drive/MyDrive/ur_fall/adl"

# Folder where processed videos will be saved (color only)
OUTPUT_FOLDER = "/content/drive/MyDrive/ur_fall/adl_color"

# ============================================================

import os

# Verify input folder exists
if os.path.exists(INPUT_FOLDER):
    # Filter ONLY videos with "cam0" (frontal view)
    all_videos = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]
    videos = [f for f in all_videos if 'cam0' in f.lower()]

    print(f"✅ Input folder found: {INPUT_FOLDER}")
    print(f"   📹 Total videos: {len(all_videos)}")
    print(f"   🎯 Cam0 videos (frontal): {len(videos)} ← ONLY THESE WILL BE PROCESSED")
    print(f"   ⏭️  Cam1 videos (ignored): {len(all_videos) - len(videos)}")
    if videos:
        print(f"\n   First cam0 files: {videos[:5]}")
else:
    print(f"❌ ERROR: Folder does not exist: {INPUT_FOLDER}")
    print("\n💡 Verify the path. You can explore your Drive in the left panel.")

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"\n✅ Output folder: {OUTPUT_FOLDER}")

## 📌 Step 3: Explore Your Drive (optional)

If you're not sure about the path, run this cell to see your Drive structure.

In [ ]:
# Explore folders in your Drive root
drive_root = "/content/drive/MyDrive"

print("📂 Folders in your Drive root:")
print("-" * 40)

for item in sorted(os.listdir(drive_root))[:20]:  # Show first 20
    full_path = os.path.join(drive_root, item)
    if os.path.isdir(full_path):
        print(f"📁 {item}")
    else:
        print(f"📄 {item}")

print("\n💡 Use these paths to configure INPUT_FOLDER above.")

## 📌 Step 4: Define Processing Functions

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import time

def remove_black_bars(frame):
    """
    Detects and removes black bars (letterbox) from the frame.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Find rows that are NOT completely black
    row_means = gray.mean(axis=1)
    threshold = 10  # Pixels with value < 10 are considered black

    non_black_rows = row_means > threshold

    # Find start and end of content
    if non_black_rows.any():
        first_row = non_black_rows.argmax()
        last_row = len(non_black_rows) - non_black_rows[::-1].argmax()
        return frame[first_row:last_row, :]

    return frame


def extract_right_half(frame):
    """
    Extracts the right half of the frame (where the color video is).
    """
    height, width = frame.shape[:2]
    return frame[:, width//2:]


def process_video(input_path, output_path):
    """
    Processes a video: removes black bars and extracts the color portion.
    """
    cap = cv2.VideoCapture(str(input_path))

    if not cap.isOpened():
        return False, "Could not open video"

    # Get properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 30  # Default
    fps = int(fps)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Read first frame to calculate final dimensions
    ret, first_frame = cap.read()
    if not ret:
        cap.release()
        return False, "Could not read first frame"

    # Process first frame to get dimensions
    no_bars = remove_black_bars(first_frame)
    color_region = extract_right_half(no_bars)

    final_height, final_width = color_region.shape[:2]

    # Configure video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(output_path), fourcc, fps, (final_width, final_height))

    # Return to beginning
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    # Process all frames
    for _ in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break

        # 1. Remove black bars
        no_bars = remove_black_bars(frame)

        # 2. Extract right half (color)
        color_frame = extract_right_half(no_bars)

        # 3. Write to output video
        out.write(color_frame)

    cap.release()
    out.release()

    return True, f"{final_width}x{final_height}"


print("✅ Processing functions loaded!")

## 📌 Step 5: Video Preview (optional)

Verify that the script correctly detects the color region.

In [ ]:
import matplotlib.pyplot as plt

# Get first video for preview
videos = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]

if videos:
    test_video = os.path.join(INPUT_FOLDER, videos[0])
    print(f"📹 Preview of: {videos[0]}")

    cap = cv2.VideoCapture(test_video)
    ret, frame = cap.read()
    cap.release()

    if ret:
        # Process frame
        no_bars = remove_black_bars(frame)
        color_only = extract_right_half(no_bars)

        # Show comparison
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Original\n{frame.shape[1]}x{frame.shape[0]}")
        axes[0].axis('off')

        axes[1].imshow(cv2.cvtColor(no_bars, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Without black bars\n{no_bars.shape[1]}x{no_bars.shape[0]}")
        axes[1].axis('off')

        axes[2].imshow(cv2.cvtColor(color_only, cv2.COLOR_BGR2RGB))
        axes[2].set_title(f"COLOR only (result)\n{color_only.shape[1]}x{color_only.shape[0]}")
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

        print("\n✅ If the third panel shows only the COLOR portion, the script is configured correctly!")
else:
    print("❌ No videos found for preview")

## 📌 Step 6: Process All Videos! 🚀

This cell processes **all** videos from the input folder and saves them to the output folder.

In [ ]:
from tqdm.notebook import tqdm
import time

print("="*65)
print("🛡️  SafeGuard Vision AI - Video Color Extractor")
print("="*65)

# Search for cam0 videos ONLY
video_extensions = ['.mp4', '.avi', '.mov', '.mkv']
videos = []

for f in os.listdir(INPUT_FOLDER):
    if any(f.lower().endswith(ext) for ext in video_extensions):
        if 'cam0' in f.lower():  # ← CAM0 FILTER
            videos.append(f)

videos = sorted(videos)

if not videos:
    print(f"\n❌ No cam0 videos found in: {INPUT_FOLDER}")
else:
    print(f"\n📂 Input:  {INPUT_FOLDER}")
    print(f"📂 Output: {OUTPUT_FOLDER}")
    print(f"🎯 Cam0 videos: {len(videos)}")
    print("-"*65)

    successful = 0
    failed = 0
    start_time = time.time()

    for i, video_name in enumerate(videos, 1):
        input_path = os.path.join(INPUT_FOLDER, video_name)
        output_name = f"color_{Path(video_name).stem}.mp4"
        output_path = os.path.join(OUTPUT_FOLDER, output_name)

        print(f"\n[{i}/{len(videos)}] 📹 {video_name}")

        success, info = process_video(input_path, output_path)

        if success:
            print(f"   ✅ Saved: {output_name} ({info})")
            successful += 1
        else:
            print(f"   ❌ Error: {info}")
            failed += 1

    # Summary
    elapsed = time.time() - start_time

    print("\n" + "="*65)
    print("📊 SUMMARY")
    print("="*65)
    print(f"   ✅ Successful:   {successful}")
    print(f"   ❌ Errors:       {failed}")
    print(f"   ⏱️  Total time:   {elapsed:.1f} seconds")
    print(f"   📂 Saved to:     {OUTPUT_FOLDER}")
    print("="*65)
    print("\n🎉 Complete! The videos are now in your Google Drive.")

## 📌 Step 7: Verify Results

List the processed videos in the output folder.

In [ ]:
print("📂 Processed videos in output folder:")
print("-" * 50)

output_videos = [f for f in os.listdir(OUTPUT_FOLDER) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]
output_videos = sorted(output_videos)

for i, video in enumerate(output_videos, 1):
    file_path = os.path.join(OUTPUT_FOLDER, video)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"   {i}. {video} ({size_mb:.1f} MB)")

print(f"\n📊 Total: {len(output_videos)} videos")
print(f"📂 Location: {OUTPUT_FOLDER}")

---

## ✅ Done!

The processed videos are now in your Google Drive in the specified folder.

**Next steps for SafeGuard Vision AI:**
1. Use these videos to extract poses with BlazePose/MediaPipe
2. Train the fall detection model
3. Evaluate with the dataset

---
*SafeGuard Vision AI - MIT Global Teaching Labs*